In [ ]:
# Instalar LightGBM com suporte a GPU
!apt-get update && \
apt-get install -y libboost-dev libboost-system-dev libboost-filesystem-dev && \
pip install lightgbm --install-option=--gpu

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,801 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,058 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,562 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,351 kB]
Get:13 http://security.ubuntu.com/ubu

In [ ]:
import pandas as pd
import numpy as np
import os # Operações de sistema de arquivos
import json # Leitura e escrita de JSON
import joblib
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb # Biblioteca de modelos GBDT
from datetime import timedelta
from google.colab import drive # Integra o Drive no ambiente Colab
import matplotlib.pyplot as plt
import time

In [ ]:
drive.mount('/content/drive') # Monta o Drive no ambiente Colab

Mounted at /content/drive


## Definição de Funções Auxiliares

In [ ]:
# Função SMAPE (Symmetric Mean Absolute Percentage Error)
def smape(y_true, y_pred):
    """
    Calcula o SMAPE entre valores verdadeiros e previstos.
    SMAPE = (1/n) * sum( |y_pred - y_true| / ((|y_true| + |y_pred|)/2) )
    Retorna porcentagem.
    """
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.mean(diff) * 100

# Função de cálculo de média móvel para nível de estoque com estoque > 0
def calc_media_leg_estoque(df, dias=7):
    """
    Adiciona coluna 'media_estoque' ao df, calculando a média móvel do estoque.
    janela: número de períodos para média.
    """
    vendas_leg = []
    for i in range(len(df)):
        inicio = max(0, i - dias)
        janela = df.iloc[inicio:i]
        janela_validade = janela[janela['estoque'] > 0]
        if not janela_validade.empty:
            media = janela_validade['venda'].mean()
        else:
            media = np.nan
        vendas_leg.append(media)
    return vendas_leg

# Diretório com arquivos CSV de treino por loja e curva
base_dir = '/content/drive/MyDrive/LINQI-grupo-1B/dados_por_loja_curva'

# lista apenas os CSVs
arquivos = [f for f in os.listdir(base_dir) if f.endswith('.csv')]

In [ ]:
!pip install lightgbm --install-option=--gpu



Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --install-option


In [ ]:
import lightgbm as lgb
print("LightGBM versão:", lgb.__version__) # verifica versão do LightGBM


LightGBM versão: 4.5.0


In [ ]:
# Grid para busca de hiperparâmetros do modelo LightGBM
param_grid = {
    'max_depth': [3, 5, 8], # Profundidade máxima da árvore
    'learning_rate': [0.05, 0.1], # Taxa de aprendizado
    'min_child_samples': [10, 20]
}

common_params = {
    'objective': 'regression',
    'metric': 'l1', # usa MAE como métrica interna
    'device': 'gpu', # utiliza GPU para otimizar tempo de treinamento
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'verbose': -1
}

# Configuração de validação cruzada temporal para séries temporais
tscv = TimeSeriesSplit(n_splits=3)

# Dicionários para armazenar resultados e melhores parâmetros
melhores_parametros = {}
avaliacoes = []

# 4. Função de tuning com lgb.cv
def tune_with_cv(dataset):
    """
    Executa lgb.cv para os parâmetros dados e retorna o melhor número de estimators.
    """
    best_score, best_combo, best_iter = float('inf'), None, None
    for combo in ParameterGrid(param_grid):
        params = {**common_params, **combo}
        cvres = lgb.cv(
            params,
            dataset,
            folds=tscv,
            num_boost_round=10000,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=0)
            ]
        )
        # pega dinamicamente a chave da métrica retornada
        metric_key = list(cvres.keys())[0]
        score = min(cvres[metric_key])
        iters = len(cvres[metric_key])
        if score < best_score:
            best_score, best_combo, best_iter = score, combo, iters
    return best_combo, best_iter

A abordagem definida pela função 'tune_with_cv' usa cross-validation interna do LightGBM para definir n_estimators ideal com early stopping.

# Pipeline Principal de Treinamento e Avaliação com Seleção de Hiperparâmetros

In [ ]:
# 5. Loop principal
for arquivo in arquivos:
    path = os.path.join(base_dir, arquivo)
    if not os.path.isfile(path):
        print(f"⚠️ Não encontrei {arquivo}, pulando.")
        continue

    # Leitura + features
    df = pd.read_csv(path)
    df['data'] = pd.to_datetime(df['data'])
    df['dia_semana_abrev'] = df['data'].dt.weekday
    df['dia_do_mes'] = df['data'].dt.day
    df['abreviacao_mes'] = df['data'].dt.month - 1
    df['fim_de_semana'] = df['dia_semana_abrev'].isin([5,6]).astype(int)
    df['inicio_fim_mes'] = ((df['dia_do_mes'] <=5)|(df['dia_do_mes']>=25)).astype(int)
    df['curva'] = df['curva'].astype('category').cat.codes
    df['dia_semana_abrev'] = df['dia_semana_abrev'].astype('category').cat.codes
    df['abreviacao_mes'] = df['abreviacao_mes'].astype('category').cat.codes
    df['preco'] = df['preco'].clip(upper=20)
    df = df.sort_values('data').copy()
    df['media_venda_leg7_estoque_pos'] = calc_media_leg_estoque(df)

    # Split 80/20: Divisão treino/teste 80/20 respeitando ordem temporal
    split = int(len(df)*0.8)
    train, test = df.iloc[:split], df.iloc[split:]
    X_train = train.drop(columns=['data','venda','estoque'])
    y_train = train['venda']
    X_test  = test.drop(columns=['data','venda','estoque'])
    y_test  = test['venda']
    if len(X_train) < 10:
        print(f"🚫 {arquivo} muito pequeno ({len(X_train)} linhas), pulando.")
        continue

    # Segmentação
    limite = np.median(y_train)
    seg_train = (y_train > limite).astype(int)
    clf = LGBMClassifier(
        n_estimators=500,
        random_state=42,
        device='gpu', gpu_platform_id=0, gpu_device_id=0
    )
    clf.fit(X_train, seg_train)
    seg_pred = clf.predict(X_test)

    # Dados por segmento (log1p no y)
    Xb = X_train[seg_train==0]; yb = np.log1p(y_train[seg_train==0])
    Xa = X_train[seg_train==1]; ya = np.log1p(y_train[seg_train==1])
    ds_b = lgb.Dataset(Xb, label=yb, free_raw_data=False)
    ds_a = lgb.Dataset(Xa, label=ya, free_raw_data=False)

    # Tuning em cada segmento
    t0 = time.time()
    best_b, it_b = tune_with_cv(ds_b)
    print(f"[{arquivo}] tuning baixo: {time.time()-t0:.1f}s, best={best_b}, rounds={it_b}")
    t0 = time.time()
    best_a, it_a = tune_with_cv(ds_a)
    print(f"[{arquivo}] tuning alto:  {time.time()-t0:.1f}s, best={best_a}, rounds={it_a}")

    # Treino final
    final_b = lgb.train({**common_params, **best_b}, ds_b, num_boost_round=it_b)
    final_a = lgb.train({**common_params, **best_a}, ds_a, num_boost_round=it_a)

    # Previsão e métricas
    y_pred = np.zeros(len(y_test))
    mask_b = (seg_pred == 0)
    y_pred[mask_b] = np.expm1(final_b.predict(X_test[mask_b]))
    y_pred[~mask_b] = np.expm1(final_a.predict(X_test[~mask_b]))

    mae = mean_absolute_error(y_test, y_pred)
    smape_val = smape(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"📊 {arquivo}: MAE={mae:.5f}, SMAPE={smape_val:.2f}%, R²={r2:.5f}")

    melhores_parametros[arquivo] = {'baixo': best_b, 'alto': best_a}
    avaliacoes.append({
        'arquivo': arquivo,
        'MAE': mae,
        'SMAPE': smape_val,
        'R2': r2
    })

# 6. Salva resultados
with open('melhores_parametros.json','w') as f:
    json.dump(melhores_parametros, f, indent=4)
pd.DataFrame(avaliacoes).to_csv('avaliacoes_modelos.csv', index=False)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:1242: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:1242: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:1242: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[586]	cv_agg's valid l1: 0.226776 + 0.00539834


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:1242: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[486]	cv_agg's valid l1: 0.227573 + 0.00513997


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:1242: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[560]	cv_agg's valid l1: 0.225384 + 0.00493524
[loja_10_curva_B.csv] tuning alto:  122.1s, best={'learning_rate': 0.1, 'max_depth': 8, 'min_child_samples': 20}, rounds=560
📊 loja_10_curva_B.csv: MAE=0.26593, SMAPE=30.66%, R²=-0.11888
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	cv_agg's valid l1: 0 + 0
Training until validation scores don't improv

In [ ]:
# mostra resultados
pd.DataFrame(avaliacoes)

,arquivo,MAE,SMAPE,R2
0,loja_1_curva_A.csv,1.003107,58.292797,0.336367
1,loja_1_curva_B.csv,0.325647,37.595926,-0.033841
2,loja_1_curva_C.csv,0.095823,14.958146,-0.081553
3,loja_1_curva_D.csv,0.022851,3.748101,-0.268761
4,loja_2_curva_A.csv,1.143004,50.064093,0.830120
5,loja_2_curva_B.csv,0.311023,34.502881,-0.027891
6,loja_2_curva_C.csv,0.087968,13.126295,-0.079647
7,loja_2_curva_D.csv,0.022276,3.463020,-0.336694
8,loja_3_curva_A.csv,0.713763,50.926636,0.656550
9,loja_3_curva_B.csv,0.213614,27.606779,-0.049406


In [ ]:
df_avaliacoes = pd.DataFrame(avaliacoes)

In [ ]:
with open('/content/drive/MyDrive/LINQI-grupo-1B/melhores_parametros.json','w') as f:
    json.dump(melhores_parametros, f, indent=4)

In [ ]:
caminho_csv = '/content/drive/MyDrive/LINQI-grupo-1B/df_avaliacoes.csv'

df_avaliacoes.to_csv(caminho_csv, index=False)
print(f'DataFrame salvo em: {caminho_csv}')

DataFrame salvo em: /content/drive/MyDrive/LINQI-grupo-1B/df_avaliacoes.csv


In [ ]:
melhores_parametros

{'loja_1_curva_A.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samples': 10},
  'alto': {'learning_rate': 0.1, 'max_depth': 8, 'min_child_samples': 20}},
 'loja_1_curva_B.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samples': 10},
  'alto': {'learning_rate': 0.1, 'max_depth': 5, 'min_child_samples': 10}},
 'loja_1_curva_C.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samples': 10},
  'alto': {'learning_rate': 0.05, 'max_depth': 5, 'min_child_samples': 10}},
 'loja_1_curva_D.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samples': 10},
  'alto': {'learning_rate': 0.1, 'max_depth': 8, 'min_child_samples': 20}},
 'loja_2_curva_A.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samples': 10},
  'alto': {'learning_rate': 0.1, 'max_depth': 5, 'min_child_samples': 20}},
 'loja_2_curva_B.csv': {'baixo': {'learning_rate': 0.05,
   'max_depth': 3,
   'min_child_samp

## Treina os todos os modelos com os respectivos hiperparametros selecionados para a amostra inicial completa

In [ ]:
# 3) Função de cálculo de média móvel (se já existir, remova/ajuste)
def calc_media_leg_estoque(df, window=7):
    return df['estoque'].rolling(window, min_periods=1).mean().shift(1)


# 5) Carrega os melhores hiperparâmetros obtidos anteriormente
with open('/content/melhores_parametros.json', 'r') as f:
    melhores_parametros = json.load(f)

# 6) Cria pasta para salvar os modelos no Drive
save_dir = '/content/drive/MyDrive/modelos'
os.makedirs(save_dir, exist_ok=True)

# 7) Loop: read, feature‐engineer, treina no dataset completo e salva
for arquivo in arquivos:
    path = os.path.join(base_dir, arquivo)
    if not os.path.isfile(path):
        print(f"⚠️  Não encontrei {arquivo}, pulando.")
        continue

    # --- a) Carrega CSV e gera features básicas ---
    df = pd.read_csv(path, parse_dates=['data'])
    df['dia_semana_abrev']  = df['data'].dt.weekday.astype('category').cat.codes
    df['dia_do_mes']        = df['data'].dt.day
    df['abreviacao_mes']    = (df['data'].dt.month - 1).astype('category').cat.codes
    df['fim_de_semana']     = df['dia_semana_abrev'].isin([5,6]).astype(int)
    df['inicio_fim_mes']    = ((df['dia_do_mes'] <= 5) | (df['dia_do_mes'] >= 25)).astype(int)
    df['curva']             = df['curva'].astype('category').cat.codes
    df['preco']             = df['preco'].clip(upper=20)
    df['media_leg7_estoque'] = calc_media_leg_estoque(df)

    # Prepara X e y completos
    X = df.drop(columns=['data','venda','estoque'])
    y = df['venda'].values

    # --- b) Segmentação: acima/abaixo da mediana de venda ---
    limite = np.median(y)
    seg = (y > limite).astype(int)

    # --- c) Treina classificador de segmento ---
    clf = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=42,
        device='gpu', gpu_platform_id=0, gpu_device_id=0
    )
    clf.fit(X, seg)

    # --- d) Separa dados para cada segmento ---
    mask_b = (seg == 0)
    X_baixo, y_baixo = X[mask_b], np.log1p(y[mask_b])
    X_alto,  y_alto  = X[~mask_b], np.log1p(y[~mask_b])

    # --- e) Recupera melhores hiperparâmetros ---
    params = melhores_parametros.get(arquivo)
    if params is None:
        print(f"🚫  Sem parâmetros para {arquivo}, pulando.")
        continue
    baixo_params = params['baixo']
    alto_params  = params['alto']

    # --- f) Treina regressors no dataset completo ---
    model_baixo = LGBMRegressor(**baixo_params, random_state=42)
    model_alto  = LGBMRegressor(**alto_params,  random_state=42)

    model_baixo.fit(X_baixo, y_baixo)
    model_alto.fit( X_alto,  y_alto)

    # --- g) Salva cada booster em arquivo de texto no Drive ---
    base_name = os.path.splitext(arquivo)[0]
    clf.booster_.save_model( os.path.join(save_dir, f'classificador_{base_name}.txt') )
    model_baixo.booster_.save_model( os.path.join(save_dir, f'modelo_baixo_{base_name}.txt') )
    model_alto.booster_.save_model(  os.path.join(save_dir, f'modelo_alto_{base_name}.txt')  )

    print(f"✔️  Modelos para {arquivo} treinados e salvos em {save_dir}")

print("🏁 Finalizado: todos os modelos foram treinados no dataset completo e salvos.")

✔️  Modelos para loja_1_curva_A.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_1_curva_B.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_1_curva_C.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_1_curva_D.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_2_curva_A.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_2_curva_B.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_2_curva_C.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_2_curva_D.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_3_curva_A.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_3_curva_B.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja_3_curva_C.csv treinados e salvos em /content/drive/MyDrive/modelos
✔️  Modelos para loja

## Realiza a predição dos dados de 22 a 28 de fevereiro para todos os modelos

In [ ]:
#  Imports e funções auxiliares
import os
import pandas as pd
import numpy as np
import lightgbm as lgb

def calc_media_leg_estoque(df, window=7):
    return df['estoque'].rolling(window, min_periods=1).mean().shift(1)

def preprocess(df):
    # supõe que exista coluna 'data' e as colunas originais de treino
    df['data'] = pd.to_datetime(df['data'])
    df['dia_semana_abrev']   = df['data'].dt.weekday.astype('category').cat.codes
    df['dia_do_mes']         = df['data'].dt.day
    df['abreviacao_mes']     = (df['data'].dt.month - 1).astype('category').cat.codes
    df['fim_de_semana']      = df['dia_semana_abrev'].isin([5,6]).astype(int)
    df['inicio_fim_mes']     = ((df['dia_do_mes'] <= 5) | (df['dia_do_mes'] >= 25)).astype(int)
    df['curva']              = df['curva'].astype('category').cat.codes
    df['preco']              = df['preco'].clip(upper=20)
    df['media_leg7_estoque'] = calc_media_leg_estoque(df)
    X = df.drop(columns=['data','venda','estoque'], errors='ignore')
    return df, X

# 3) Defina diretórios
base_dir = '/content/drive/MyDrive/LINQI-grupo-1B/dados_por_loja_curva_valid'
models_dir = '/content/drive/My Drive/modelos'
output_dir = '/content/drive/MyDrive/LINQI-grupo-1B/predicoes'
os.makedirs(output_dir, exist_ok=True)

# 4) Carrega e prediz para cada arquivo
for fname in os.listdir(base_dir):
    if not fname.endswith('.csv'):
        continue

    path = os.path.join(base_dir, fname)
    df = pd.read_csv(path)
    df_proc, X = preprocess(df)

    base_name = os.path.splitext(fname)[0]

    # 4.1) Carrega classificador de segmento
    clf_booster = lgb.Booster(model_file=os.path.join(models_dir, f'classificador_{base_name}.txt'))
    seg_raw = clf_booster.predict(X)
    seg = (seg_raw > 0.5).astype(int)

    # 4.2) Carrega regressors de cada segmento
    model_b = lgb.Booster(model_file=os.path.join(models_dir, f'modelo_baixo_{base_name}.txt'))
    model_a = lgb.Booster(model_file=os.path.join(models_dir, f'modelo_alto_{base_name}.txt'))

    # 4.3) Gera predições
    preds = np.zeros(len(X), dtype=float)
    mask_b = (seg == 0)
    if mask_b.any():
        preds[mask_b] = np.expm1(model_b.predict(X[mask_b]))
    if (~mask_b).any():
        preds[~mask_b] = np.expm1(model_a.predict(X[~mask_b]))

    # 4.4) Salva resultados
    df_proc['pred_venda'] = preds
    out_path = os.path.join(output_dir, f'pred_{base_name}.csv')
    df_proc.to_csv(out_path, index=False)
    print(f'✔️  Predições salvas em: {out_path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_3_curva_D.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_4_curva_A.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_5_curva_C.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_4_curva_B.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_3_curva_A.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_5_curva_A.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_2_curva_C.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_3_curva_B.csv
✔️  Predições salvas em: /content/drive/MyDrive/LINQI-grupo-1B/predicoes/pred_loja_2_curva_B.cs

## Avalia os desempenhos das predições pelas métricas MAE e SMAPE

In [ ]:


# 2) Imports
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# 3) Defina função SMAPE
def smape(y_true, y_pred):
    # evita divisão por zero
    denom = (np.abs(y_true) + np.abs(y_pred))
    # quando denom == 0, definimos termo SMAPE como 0
    nonzero = denom != 0
    sm = np.zeros_like(y_true, dtype=float)
    sm[nonzero] = 2 * np.abs(y_pred[nonzero] - y_true[nonzero]) / denom[nonzero]
    return np.mean(sm) * 100

# 4) Diretórios
pred_dir   = '/content/drive/MyDrive/LINQI-grupo-1B/predicoes'
eval_path  = '/content/drive/MyDrive/LINQI-grupo-1B/avaliacao_predicoes.csv'

# 5) Avaliação
resultados = []

for fname in os.listdir(pred_dir):
    if not fname.startswith('pred_') or not fname.endswith('.csv'):
        continue
    path = os.path.join(pred_dir, fname)
    df = pd.read_csv(path)
    if 'venda' not in df.columns or 'pred_venda' not in df.columns:
        print(f"⚠️  Colunas faltando em {fname}, pulando.")
        continue
    y_true = df['venda'].values
    y_pred = df['pred_venda'].values

    mae_val   = mean_absolute_error(y_true, y_pred)
    smape_val = smape(y_true, y_pred)

    resultados.append({
        'arquivo': fname.replace('pred_',''),
        'MAE': mae_val,
        'SMAPE_%': smape_val
    })
    print(f"✅ {fname}: MAE={mae_val:.5f}, SMAPE={smape_val:.2f}%")

# 6) Salva avaliação no Drive
df_eval = pd.DataFrame(resultados)
df_eval.to_csv(eval_path, index=False)
print(f"\n🏁 Avaliação salva em: {eval_path}")


✅ pred_loja_3_curva_D.csv: MAE=0.01387, SMAPE=2.28%
✅ pred_loja_4_curva_A.csv: MAE=0.62065, SMAPE=46.14%
✅ pred_loja_5_curva_C.csv: MAE=0.06851, SMAPE=11.00%
✅ pred_loja_4_curva_B.csv: MAE=0.15093, SMAPE=20.47%
✅ pred_loja_3_curva_A.csv: MAE=0.86998, SMAPE=53.98%
✅ pred_loja_5_curva_A.csv: MAE=1.19422, SMAPE=52.53%
✅ pred_loja_2_curva_C.csv: MAE=0.07616, SMAPE=12.09%
✅ pred_loja_3_curva_B.csv: MAE=0.20676, SMAPE=27.85%
✅ pred_loja_2_curva_B.csv: MAE=0.29464, SMAPE=34.49%
✅ pred_loja_1_curva_A.csv: MAE=1.07955, SMAPE=66.04%
✅ pred_loja_2_curva_A.csv: MAE=1.58929, SMAPE=51.09%
✅ pred_loja_1_curva_B.csv: MAE=0.34754, SMAPE=40.16%
✅ pred_loja_4_curva_D.csv: MAE=0.01319, SMAPE=1.99%
✅ pred_loja_3_curva_C.csv: MAE=0.05278, SMAPE=8.89%
✅ pred_loja_1_curva_C.csv: MAE=0.08647, SMAPE=13.61%
✅ pred_loja_2_curva_D.csv: MAE=0.02117, SMAPE=2.86%
✅ pred_loja_5_curva_B.csv: MAE=0.24116, SMAPE=30.86%
✅ pred_loja_1_curva_D.csv: MAE=0.01789, SMAPE=3.05%
✅ pred_loja_4_curva_C.csv: MAE=0.04455, SMAPE=7.14%

In [ ]:


# 1. Real vs Predito
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel("Valor Real")
plt.ylabel("Valor Predito")
plt.title("Real vs Predito - Loja 1 Curva C")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Distribuição dos Resíduos
residuos = y_test - y_pred
plt.figure(figsize=(8,6))
plt.hist(residuos, bins=50, edgecolor='black')
plt.title("Distribuição dos Resíduos - Loja 1 Curva C")
plt.xlabel("Erro (Real - Predito)")
plt.ylabel("Frequência")
plt.grid(True)
plt.tight_layout()
plt.show()

# 3. Erro Absoluto vs Valor Real
erro_absoluto = np.abs(y_test - y_pred)
plt.figure(figsize=(8,6))
plt.scatter(y_test, erro_absoluto, alpha=0.4)
plt.xlabel("Valor Real")
plt.ylabel("Erro Absoluto")
plt.title("Erro Absoluto vs Valor Real - Loja 1 Curva C")
plt.grid(True)
plt.tight_layout()
plt.show()

# 4. Importância das Variáveis

def plot_importancias(modelo, titulo, top_n=10):
    importancias = pd.Series(modelo.feature_importances_, index=X_train.columns)
    importancias = importancias.sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(8,6))
    sns.barplot(x=importancias, y=importancias.index)
    plt.title(titulo)
    plt.xlabel("Importância")
    plt.tight_layout()
    plt.show()

plot_importancias(modelo_baixo, "Importância das Variáveis - Preço Baixo")
plot_importancias(modelo_alto, "Importância das Variáveis - Preço Alto")

In [ ]:
# 1. Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Importar bibliotecas
import pandas as pd
import joblib  # ou lightgbm se for Booster
import matplotlib.pyplot as plt

# 3. Definir caminhos
# Ajuste o caminho conforme a sua organização de pastas no Drive
MODEL_PATH = '/content/drive/MyDrive/LINQI-grupo-1B/modelos/modelo_loja_1_curva_C.txt'
TEST_DATA_PATH = '/content/drive/MyDrive/LINQI-grupo-1B/dados_por_loja_curva_valid/loja_1_curva_C.csv'

# 4. Carregar o modelo
# Se você usou joblib:
model = joblib.load(MODEL_PATH)
# Se for um Booster do LightGBM:
# import lightgbm as lgb
# model = lgb.Booster(model_file=MODEL_PATH)

# 5. Carregar e pré-processar os dados de teste
df_test = pd.read_csv(TEST_DATA_PATH, parse_dates=['data'])
# Criar mesmas features usadas no treino
df_test['dia_semana']     = df_test['data'].dt.weekday
df_test['dia_do_mes']     = df_test['data'].dt.day
df_test['mes_abreviado']  = df_test['data'].dt.month - 1
# (adicione aqui outras colunas/encodings que o seu pipeline de treino usou)

# 6. Separar X e y
X_test = df_test.drop(columns=['vendas', 'data'])  # ‘vendas’ é a coluna alvo
y_test = df_test['vendas']

# 7. Fazer a previsão
y_pred = model.predict(X_test)

# 8. Anexar previsões ao DataFrame
df_test['predito'] = y_pred

# 9. Gráfico 1: scatter real × predito
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2)
plt.xlabel('Vendas Reais')
plt.ylabel('Vendas Preditas')
plt.title('Curva C – Loja 1: Real vs. Predito')
plt.grid(True)
plt.show()

# 10. Gráfico 2: série temporal real e predito
plt.figure(figsize=(12, 6))
plt.plot(df_test['data'], y_test,    label='Real')
plt.plot(df_test['data'], y_pred,    label='Predito', alpha=0.7)
plt.xlabel('Data')
plt.ylabel('Vendas')
plt.title('Curva C – Loja 1: Vendas Reais x Preditas ao Longo do Tempo')
plt.legend()
plt.grid(True)
plt.show()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/LINQI-grupo-1B/modelos/modelo_loja_1_curva_C.txt'

In [ ]:
# 1. Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Importar bibliotecas
import pandas as pd
import joblib   # ou lightgbm se você salvou como Booster
import matplotlib.pyplot as plt

# 3. Definir caminhos dos modelos e dos dados de teste
MODEL_PATH_BAIXO = '/content/drive/MyDrive/LINQI-grupo-1B/modelos/modelo_baixo_loja_1_curva_C.txt'
MODEL_PATH_ALTO  = '/content/drive/MyDrive/LINQI-grupo-1B/modelos/modelo_alto_loja_1_curva_C.txt'
TEST_DATA_PATH   = '/content/drive/MyDrive/LINQI-grupo-1B/dados_por_loja_curva_valid/loja_1_curva_C.csv'

# 4. Carregar os modelos
modelo_baixo = joblib.load(MODEL_PATH_BAIXO)
modelo_alto  = joblib.load(MODEL_PATH_ALTO)
# se for Booster do LightGBM, troque por:
# import lightgbm as lgb
# modelo_baixo = lgb.Booster(model_file=MODEL_PATH_BAIXO)
# modelo_alto  = lgb.Booster(model_file=MODEL_PATH_ALTO)

# 5. Carregar e pré-processar dados de teste
df = pd.read_csv(TEST_DATA_PATH, parse_dates=['data'])
df['dia_semana']    = df['data'].dt.weekday
df['dia_do_mes']    = df['data'].dt.day
df['mes_abreviado'] = df['data'].dt.month - 1
# ... inclua aqui quaisquer outras features do pipeline de treino ...

X = df.drop(columns=['vendas', 'data',estoque])
y = df['vendas']

# 6. Prever com cada modelo
df['predito_baixo'] = modelo_baixo.predict(X)
df['predito_alto']  = modelo_alto.predict(X)

# 7. Função auxiliar para plotar
def plot_results(y_true, y_pred, title_suffix):
    # Scatter real × predito
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, 'r--', linewidth=2)
    plt.xlabel('Vendas Reais')
    plt.ylabel('Vendas Preditas')
    plt.title(f'Curva C – Loja 1 ({title_suffix}): Real vs. Predito')
    plt.grid(True)
    plt.show()

    # Série temporal real e predito
    plt.figure(figsize=(12, 6))
    plt.plot(df['data'], y_true, label='Real')
    plt.plot(df['data'], y_pred, label='Predito', alpha=0.7)
    plt.xlabel('Data')
    plt.ylabel('Vendas')
    plt.title(f'Curva C – Loja 1 ({title_suffix}): Vendas Reais x Preditas')
    plt.legend()
    plt.grid(True)
    plt.show()

# 8. Gerar os gráficos para "baixo preço"
plot_results(y, df['predito_baixo'], 'Baixo Preço')

# 9. Gerar os gráficos para "alto preço"
plot_results(y, df['predito_alto'], 'Alto Preço')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


IndexError: pop from empty list